In [1]:
import os
import re
import json
import logging
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
from collections import Counter
from functools import partial
import random


import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns


from tqdm.auto import tqdm


import joblib
import pickle


from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score, classification_report,
    precision_recall_curve, confusion_matrix
)


from scipy import sparse


try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
except Exception:
    MultilabelStratifiedKFold = None
    MultilabelStratifiedShuffleSplit = None


try:

    from gensim.models import KeyedVectors
except Exception:
    KeyedVectors = None

try:
    import fasttext
    import fasttext.util
except Exception:
    fasttext = None


try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW
except Exception:
    torch = None


try:
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        TrainingArguments, Trainer, DataCollatorWithPadding
    )
except Exception:
    AutoTokenizer = AutoModelForSequenceClassification = TrainingArguments = Trainer = DataCollatorWithPadding = None


import warnings
warnings.filterwarnings("ignore")


sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

C:\Users\valentin\.conda\envs\mirea-torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed: int = 42) -> None:
    # Фиксируем seed для воспроизводимости (насколько это возможно).
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

    # Настройки для более детерминированного поведения на GPU (если доступно).
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

    # Включаем детерминированные алгоритмы (PyTorch >=1.8). При необходимости окружение CUDA
    # может требовать настройки переменных CUBLAS_WORKSPACE_CONFIG или CPU реализаций.
    # try:
    #     torch.use_deterministic_algorithms(True)
    # except Exception:
    #     try:
    #         torch.set_deterministic(True)
    #     except Exception:
    #         pass

    # Сделаем генератор для DataLoader доступным глобально, чтобы все загрузчики были детерминированы
    global DATA_LOADER_GEN
    try:
        DATA_LOADER_GEN = torch.Generator()
        DATA_LOADER_GEN.manual_seed(seed)
    except Exception:
        DATA_LOADER_GEN = None

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [3]:


# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.tsv', 'validation': 'dev.tsv'}
df_train = pd.read_csv("hf://datasets/s-nlp/ru_paradetox/" + splits["train"], sep="\t")
df_val = pd.read_csv("hf://datasets/s-nlp/ru_paradetox/" + splits["validation"], sep="\t")

In [4]:
df_train.head()

,ru_toxic_comment,ru_neutral_comment
0,"и,чё,блядь где этот херой был до этого со свои...","Ну и где этот герой был,со своими доказательст..."
1,"и,чё,блядь где этот херой был до этого со свои...",Где этот герой был до этого со своими доказате...
2,"и,чё,блядь где этот херой был до этого со свои...","и,где этот герой был до этого со своими доказа..."
3,"О, а есть деанон этого петуха?","О, а есть деанон"
4,"херну всякую пишут,из-за этого лайка.долбоебизм.","Чушь всякую пишут, из- за этого лайка."


In [5]:
df_val.head()

,ru_toxic_comment,ru_neutral_comment
0,пиздеж! температуры горения хватит чтобы её ра...,Враньё! Температуры горения хватит чтобы ее ра...
1,пиздеж! температуры горения хватит чтобы её ра...,"неправда,температуры горения хватит чтобы расп..."
2,пиздеж! температуры горения хватит чтобы её ра...,Враньё! Температуры горения хватит на чтобы её...
3,а ты чмо там был.ты вообще служил.гандон,А ты там был? Ты вообще служил?
4,пиздабол ---- а сам где кормишься ?,а сам где кормишься ?


In [6]:
df_train.shape

(11090, 2)

In [7]:
df_val.shape

(1116, 2)

In [8]:
df_train = df_train.drop_duplicates().reset_index(drop=True)
df_val = df_val.drop_duplicates().reset_index(drop=True)

mask_error_train = df_train.astype(str).apply(lambda col: col.str.contains(r"#ERROR!", na=False)).any(axis=1)
mask_error_val = df_val.astype(str).apply(lambda col: col.str.contains(r"#ERROR!", na=False)).any(axis=1)

df_train = df_train.loc[~mask_error_train].reset_index(drop=True)
df_val = df_val.loc[~mask_error_val].reset_index(drop=True)


In [9]:
df_train.shape

(10882, 2)

In [10]:
df_val.shape

(1087, 2)

In [11]:
df_train = df_train.sample(frac=0.1, random_state=42).reset_index(drop=True)
df_val = df_val.sample(frac=0.1, random_state=42).reset_index(drop=True)


In [12]:
df_train, df_test = train_test_split(df_train, test_size=0.1, random_state=42)
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


In [13]:
max_len = 40
vocab_size = 20000
special_tokens = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]

all_texts = pd.concat([
    df_train["ru_toxic_comment"].astype(str),
    df_train["ru_neutral_comment"].astype(str)
], ignore_index=True)

counter = Counter()
for text in all_texts:
    counter.update(text.lower().split())

most_common = [w for w, _ in counter.most_common(vocab_size - len(special_tokens))]
idx2word = special_tokens + most_common
word2idx = {w: i for i, w in enumerate(idx2word)}

pad_id = word2idx["<PAD>"]
bos_id = word2idx["<BOS>"]
eos_id = word2idx["<EOS>"]
unk_id = word2idx["<UNK>"]


In [14]:
def encode_text(text, max_len):
    tokens = [word2idx.get(w, unk_id) for w in str(text).lower().split()]
    tokens = [bos_id] + tokens[:max_len - 2] + [eos_id]
    if len(tokens) < max_len:
        tokens = tokens + [pad_id] * (max_len - len(tokens))
    return tokens

def make_arrays(df):
    src = np.array([encode_text(t, max_len) for t in df["ru_toxic_comment"].astype(str)])
    tgt = np.array([encode_text(t, max_len) for t in df["ru_neutral_comment"].astype(str)])
    tgt_in = tgt[:, :-1]
    tgt_out = tgt[:, 1:]
    return src, tgt_in, tgt_out


In [15]:
X_train_src, X_train_tgt_in, X_train_tgt_out = make_arrays(df_train)
X_val_src, X_val_tgt_in, X_val_tgt_out = make_arrays(df_val)
X_test_src, X_test_tgt_in, X_test_tgt_out = make_arrays(df_test)


In [16]:
class Seq2SeqDataset(Dataset):
    def __init__(self, src, tgt_in, tgt_out):
        self.src = torch.LongTensor(src)
        self.tgt_in = torch.LongTensor(tgt_in)
        self.tgt_out = torch.LongTensor(tgt_out)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt_in[idx], self.tgt_out[idx]


In [17]:
class RNNSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.RNN(embed_dim, hidden_size, batch_first=True)
        self.decoder = nn.RNN(embed_dim, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, src, tgt_in):
        src_emb = self.embedding(src)
        _, h = self.encoder(src_emb)
        tgt_emb = self.embedding(tgt_in)
        dec_out, _ = self.decoder(tgt_emb, h)
        logits = self.output(dec_out)
        return logits

class GRUSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, src, tgt_in):
        src_emb = self.embedding(src)
        _, h = self.encoder(src_emb)
        tgt_emb = self.embedding(tgt_in)
        dec_out, _ = self.decoder(tgt_emb, h)
        logits = self.output(dec_out)
        return logits

class LSTMSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, src, tgt_in):
        src_emb = self.embedding(src)
        _, (h, c) = self.encoder(src_emb)
        tgt_emb = self.embedding(tgt_in)
        dec_out, _ = self.decoder(tgt_emb, (h, c))
        logits = self.output(dec_out)
        return logits


In [18]:
def train_seq2seq(model, train_ds, val_ds, epochs=3, lr=1e-3, batch_size=32):
    model = model.to(device)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, generator=DATA_LOADER_GEN)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    optimizer = AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id)
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for src, tgt_in, tgt_out in train_loader:
            src = src.to(device)
            tgt_in = tgt_in.to(device)
            tgt_out = tgt_out.to(device)
            optimizer.zero_grad()
            logits = model(src, tgt_in)
            loss = loss_fn(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        model.eval()
        with torch.no_grad():
            val_loss = 0
            for src, tgt_in, tgt_out in val_loader:
                src = src.to(device)
                tgt_in = tgt_in.to(device)
                tgt_out = tgt_out.to(device)
                logits = model(src, tgt_in)
                loss = loss_fn(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
                val_loss += loss.item()
        print(f"epoch {epoch+1} train_loss {total_loss/len(train_loader):.4f} val_loss {val_loss/len(val_loader):.4f}")
    return model


In [19]:
def greedy_decode(model, src, max_len):
    model.eval()
    src = src.to(device)
    src_emb = model.embedding(src)
    if isinstance(model, LSTMSeq2Seq):
        _, (h, c) = model.encoder(src_emb)
    else:
        _, h = model.encoder(src_emb)
    decoded = torch.full((src.size(0), 1), bos_id, dtype=torch.long, device=src.device)
    for _ in range(max_len - 1):
        emb = model.embedding(decoded)
        if isinstance(model, LSTMSeq2Seq):
            dec_out, (h, c) = model.decoder(emb, (h, c))
        else:
            dec_out, h = model.decoder(emb, h)
        logits = model.output(dec_out[:, -1:, :])
        next_token = logits.argmax(dim=-1)
        decoded = torch.cat([decoded, next_token], dim=1)
    return decoded


In [20]:
def decode_sequences(seqs):
    texts = []
    for seq in seqs:
        words = []
        for idx in seq:
            if idx == eos_id:
                break
            if idx in (pad_id, bos_id):
                continue
            words.append(idx2word[idx] if idx < len(idx2word) else "")
        texts.append(" ".join(words).strip())
    return texts


In [21]:
train_ds = Seq2SeqDataset(X_train_src, X_train_tgt_in, X_train_tgt_out)
val_ds = Seq2SeqDataset(X_val_src, X_val_tgt_in, X_val_tgt_out)


In [22]:
model_rnn = train_seq2seq(RNNSeq2Seq(len(idx2word)), train_ds, val_ds, epochs=20, lr=1e-3, batch_size=32)



epoch 1 train_loss 8.4608 val_loss 8.1236
epoch 2 train_loss 6.9508 val_loss 8.0559
epoch 3 train_loss 6.6258 val_loss 8.0403
epoch 4 train_loss 6.4518 val_loss 8.0345
epoch 5 train_loss 6.2813 val_loss 8.1042
epoch 6 train_loss 6.0964 val_loss 8.1064
epoch 7 train_loss 5.9107 val_loss 8.2020
epoch 8 train_loss 5.7104 val_loss 8.1350
epoch 9 train_loss 5.5066 val_loss 8.2076
epoch 10 train_loss 5.2888 val_loss 8.2361
epoch 11 train_loss 5.0709 val_loss 8.3002
epoch 12 train_loss 4.8468 val_loss 8.2886
epoch 13 train_loss 4.6254 val_loss 8.4435
epoch 14 train_loss 4.4027 val_loss 8.5535
epoch 15 train_loss 4.1840 val_loss 8.5283
epoch 16 train_loss 3.9678 val_loss 8.6741
epoch 17 train_loss 3.7541 val_loss 8.7502
epoch 18 train_loss 3.5472 val_loss 8.8230
epoch 19 train_loss 3.3511 val_loss 8.8232
epoch 20 train_loss 3.1572 val_loss 8.9988


In [23]:
model_gru = train_seq2seq(GRUSeq2Seq(len(idx2word)), train_ds, val_ds, epochs=3, lr=1e-3, batch_size=32)


epoch 1 train_loss 8.3560 val_loss 7.5652
epoch 2 train_loss 6.9063 val_loss 7.8603
epoch 3 train_loss 6.6724 val_loss 8.0129


In [24]:
model_lstm = train_seq2seq(LSTMSeq2Seq(len(idx2word)), train_ds, val_ds, epochs=3, lr=1e-3, batch_size=32)

epoch 1 train_loss 8.4229 val_loss 7.7699
epoch 2 train_loss 6.9454 val_loss 7.8538
epoch 3 train_loss 6.6871 val_loss 8.0002


In [25]:
import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup,
)

from tqdm.auto import tqdm

modelname = "cointegrated/rut5-small"
maxlen = 128

batchsize = 8
numepochs = 2
lr = 5e-5
logsteps = 50
seed = 42

torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(modelname, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(modelname).to(device)

class DetoxSeq2SeqDataset(Dataset):
    def __init__(self, df, tokenizer, maxlen):
        self.inputs = tokenizer(
            ["detoxify: " + str(x) for x in df["ru_toxic_comment"].tolist()],
            truncation=True,
            padding="max_length",
            max_length=maxlen,
            return_tensors="pt",
        )

        labels = tokenizer(
            df["ru_neutral_comment"].astype(str).tolist(),
            truncation=True,
            padding="max_length",
            max_length=maxlen,
            return_tensors="pt",
        )["input_ids"]

        labels[labels == tokenizer.pad_token_id] = -100
        self.labels = labels

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.inputs.items()}
        item["labels"] = self.labels[idx]
        return item

train_ds = DetoxSeq2SeqDataset(df_train, tokenizer, maxlen)
val_ds = DetoxSeq2SeqDataset(df_val, tokenizer, maxlen)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

train_loader = DataLoader(
    train_ds,
    batch_size=batchsize,
    shuffle=True,
    collate_fn=collator,
)

val_loader = DataLoader(
    val_ds,
    batch_size=batchsize,
    shuffle=False,
    collate_fn=collator,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

num_training_steps = numepochs * len(train_loader)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

use_fp16 = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_fp16)

def eval_loss(model, loader):
    model.eval()
    total = 0.0
    n = 0

    with torch.no_grad():
        pbar = tqdm(loader, desc="eval", leave=False)

        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}

            with torch.cuda.amp.autocast(enabled=use_fp16):
                out = model(**batch)
                loss = out.loss

            total += loss.item()
            n += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total / max(1, n)

global_step = 0

for epoch in range(numepochs):
    model.train()

    pbar = tqdm(train_loader, desc=f"train {epoch+1}/{numepochs}")

    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_fp16):
            out = model(**batch)
            loss = out.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        global_step += 1

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}",
        )

        if global_step % logsteps == 0:
            print(f"step={global_step} loss={loss.item():.4f}")

    val_loss = eval_loss(model, val_loader)
    print(f"epoch={epoch+1} val_loss={val_loss:.4f}")

class DetoxGenDataset(Dataset):
    def __init__(self, df, tokenizer, maxlen):
        self.inputs = tokenizer(
            ["detoxify: " + str(x) for x in df["ru_toxic_comment"].tolist()],
            truncation=True,
            padding="max_length",
            max_length=maxlen,
            return_tensors="pt",
        )

    def __len__(self):
        return self.inputs["input_ids"].size(0)

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.inputs.items()}

gen_ds = DetoxGenDataset(df_val, tokenizer, maxlen)
gen_loader = DataLoader(gen_ds, batch_size=batchsize, shuffle=False)

model.eval()
preds = []

with torch.no_grad():
    pbar = tqdm(gen_loader, desc="generate")

    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=64,
            num_beams=4,
        )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        preds.extend([x.strip() for x in decoded])

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
train 1/2:  41%|████      | 50/123 [01:02<01:28,  1.21s/it, loss=nan, lr=3.98e-05]   

step=50 loss=nan


train 1/2:  81%|████████▏ | 100/123 [02:02<00:27,  1.21s/it, loss=nan, lr=2.97e-05]  

step=100 loss=nan


train 1/2: 100%|██████████| 123/123 [02:29<00:00,  1.22s/it, loss=nan, lr=2.50e-05]   


...

train 2/2:  63%|██████▎   | 77/123 [01:33<00:55,  1.21s/it, loss=nan, lr=9.35e-06]   

step=200 loss=nan


train 2/2: 100%|██████████| 123/123 [02:28<00:00,  1.21s/it, loss=nan, lr=0.00e+00]


epoch=2 val_loss=nan


generate: 100%|██████████| 14/14 [00:30<00:00,  2.21s/it]


In [ ]:
val_src_tensor = torch.LongTensor(X_val_src)

preds_rnn = decode_sequences(greedy_decode(model_rnn, val_src_tensor, max_len).cpu().numpy())
preds_gru = decode_sequences(greedy_decode(model_gru, val_src_tensor, max_len).cpu().numpy())
preds_lstm = decode_sequences(greedy_decode(model_lstm, val_src_tensor, max_len).cpu().numpy())

refs = df_val["ru_neutral_comment"].astype(str).tolist()


In [27]:
try:
    import evaluate
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    bertscore = evaluate.load("bertscore")
except Exception as e:
    raise ImportError("Install evaluate to compute BLEU/ROUGE/BERTScore") from e


In [28]:
results = []
for name, preds in [("RNN", preds_rnn), ("GRU", preds_gru), ("LSTM", preds_lstm), ("Transformer", preds_transformer)]:
    bleu_score = bleu.compute(predictions=preds, references=refs)["bleu"]
    rouge_scores = rouge.compute(predictions=preds, references=refs)
    bert_scores = bertscore.compute(predictions=preds, references=refs, model_type="DeepPavlov/rubert-base-cased")
    results.append({
        "model": name,
        "bleu": bleu_score,
        "rouge1": rouge_scores.get("rouge1"),
        "rouge2": rouge_scores.get("rouge2"),
        "rougeL": rouge_scores.get("rougeL"),
        "bertscore_f1": float(np.mean(bert_scores["f1"]))
    })

metrics_df = pd.DataFrame(results)
print(metrics_df)


NameError: name 'preds_transformer' is not defined